# Cross-Lingual Retrieval & Multilingual QA

## 1. Project Overview

**Task:** Build a QA system that retrieves documents written in **one language** and generates answers in a **different language**. The knowledge base contains documents in English, Spanish, and French — but users can ask questions and receive answers in any of those languages.

**Why this matters:** Most of the world's knowledge is locked in English. A Spanish-speaking user searching for technical documentation shouldn't need to know English to find and understand the answer. Cross-lingual retrieval lets you:
- Search English docs with Spanish queries (and vice versa)
- Answer in the user's language regardless of the document's language
- Build truly multilingual knowledge bases

**Key concepts covered:**
1. **Multilingual embeddings** — how models like `paraphrase-multilingual-MiniLM-L12-v2` map text from different languages into the same vector space
2. **Cross-lingual retrieval** — querying in one language, matching documents in another
3. **Answer language control** — generating responses in the user's preferred language

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — LLM for answer generation (supports multilingual output)
- `HuggingFaceEmbeddings` (`paraphrase-multilingual-MiniLM-L12-v2`) — multilingual embedding model
- `ChromaDB` — vector store

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **how multilingual embeddings work** — shared vector space across languages |
| 2 | See **why monolingual embeddings fail** for cross-lingual retrieval |
| 3 | Build a **cross-lingual retrieval pipeline** — query in language A, retrieve in language B |
| 4 | Implement **answer-language control** — respond in the user's preferred language |
| 5 | Evaluate **retrieval quality across language pairs** |
| 6 | Compare **monolingual vs multilingual** embedding performance |

## 3. How Multilingual Embeddings Work

### The Problem with Monolingual Embeddings

Standard English embedding models (like `all-MiniLM-L6-v2`) map English text to vectors. If you embed Spanish text with them, the vectors land in a **completely different region** of the embedding space:

```
MONOLINGUAL EMBEDDING SPACE:

  English cluster                    Spanish cluster
  ┌──────────────┐                   ┌──────────────┐
  │ "cat"    ●   │                   │ "gato"   ●   │
  │ "dog"    ●   │    far apart!     │ "perro"  ●   │
  │ "house"  ●   │   ←──────────→    │ "casa"   ●   │
  └──────────────┘                   └──────────────┘

  cos_sim("cat", "gato") ≈ 0.15  ← Terrible! Same meaning, low similarity.
```

### The Multilingual Solution

Multilingual embedding models are trained on **parallel text** (same sentences in multiple languages) so that semantically equivalent text gets **similar vectors regardless of language**:

```
MULTILINGUAL EMBEDDING SPACE:

  Shared semantic space
  ┌────────────────────────────────────┐
  │         "cat" ● ● "gato"           │  ← Same region!
  │               ● "chat"             │
  │                                    │
  │   "dog" ● ● "perro"                │  ← Same region!
  │             ● "chien"              │
  │                                    │
  │        "house" ● ● "casa"          │  ← Same region!
  │                  ● "maison"        │
  └────────────────────────────────────┘

  cos_sim("cat", "gato")  ≈ 0.85  ← 
  cos_sim("cat", "chat")  ≈ 0.82  ←  Cross-lingual retrieval works!
  cos_sim("cat", "perro") ≈ 0.30  ← Different meaning, low similarity.
```

### Model: `paraphrase-multilingual-MiniLM-L12-v2`

- Trained on 50+ languages using parallel data
- 384-dimensional embeddings (same as the monolingual MiniLM)
- Designed so that translations of the same sentence cluster together
- Slight quality trade-off vs monolingual models for same-language retrieval

## 4. Pipeline Architecture

```
  User Question (any language)
     │
     ├── Detect language
     │
     ▼
  ┌──────────────────────────┐
  │  Multilingual Embedding  │  Same model for all languages
  │  (query → vector)        │
  └──────────┬───────────────┘
             │
             ▼
  ┌──────────────────────────┐
  │  ChromaDB Vector Search  │  Matches against docs in ANY language
  │  (cosine similarity)     │
  └──────────┬───────────────┘
             │
             ▼  Retrieved docs (may be in different language)
  ┌──────────────────────────┐
  │  LLM Answer Generation   │  System prompt: "Answer in {user_lang}"
  │  with language control   │
  └──────────────────────────┘
             │
             ▼
     Answer in user's language
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers

print("Dependencies: langchain, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")
print("Multilingual embedding: paraphrase-multilingual-MiniLM-L12-v2")

## 6. Imports & Configuration

In [ ]:
import os
import re
import json
import time
import textwrap
import warnings
from collections import Counter

import numpy as np

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

LLM_MODEL = "qwen3.5:9b"
MULTILINGUAL_EMBED = "paraphrase-multilingual-MiniLM-L12-v2"
MONOLINGUAL_EMBED = "all-MiniLM-L6-v2"
TOP_K = 3

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Multilingual embeddings: {MULTILINGUAL_EMBED}")
print(f"Monolingual embeddings: {MONOLINGUAL_EMBED} (baseline)")

## 7. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Quick LLM check — multilingual
test = ask("Diga 'listo' en español.")
print(f"LLM check: {test[:40]}")

---

# Part A — Multilingual Embedding Exploration

## 8. Load Both Embedding Models

We load the monolingual (English-only) and multilingual models side by side to compare how they handle cross-lingual pairs.

In [ ]:
embed_mono = HuggingFaceEmbeddings(model_name=MONOLINGUAL_EMBED)
embed_multi = HuggingFaceEmbeddings(model_name=MULTILINGUAL_EMBED)

print(f"Monolingual model: {MONOLINGUAL_EMBED}")
print(f"Multilingual model: {MULTILINGUAL_EMBED}")

# Quick dimension check
test_vec_mono = embed_mono.embed_query("hello")
test_vec_multi = embed_multi.embed_query("hello")
print(f"Monolingual dims: {len(test_vec_mono)}")
print(f"Multilingual dims: {len(test_vec_multi)}")

## 9. Cross-Lingual Similarity — Monolingual vs Multilingual

In [ ]:
# Translation pairs — same meaning in different languages
translation_pairs = [
    ("The company's revenue grew by 15% this quarter.",
     "Los ingresos de la empresa crecieron un 15% este trimestre.",
     "Le chiffre d'affaires de l'entreprise a augmenté de 15% ce trimestre."),

    ("The new security policy requires multi-factor authentication.",
     "La nueva política de seguridad requiere autenticación multifactor.",
     "La nouvelle politique de sécurité exige une authentification multifacteur."),

    ("Remote work is allowed three days per week.",
     "El trabajo remoto está permitido tres días por semana.",
     "Le télétravail est autorisé trois jours par semaine."),

    ("Machine learning models must be retrained quarterly.",
     "Los modelos de aprendizaje automático deben ser reentrenados trimestralmente.",
     "Les modèles d'apprentissage automatique doivent être réentraînés chaque trimestre."),
]

print("CROSS-LINGUAL SIMILARITY COMPARISON")
print("=" * 90)
print(f"{'Pair':<8} {'Lang Pair':<10} {'Monolingual':<14} {'Multilingual':<14} {'Delta':>8}")
print("-" * 54)

mono_scores = []
multi_scores = []

for idx, (en, es, fr) in enumerate(translation_pairs, 1):
    for lang_pair, (text_a, text_b) in [("EN→ES", (en, es)), ("EN→FR", (en, fr)), ("ES→FR", (es, fr))]:
        # Monolingual
        va_mono = embed_mono.embed_query(text_a)
        vb_mono = embed_mono.embed_query(text_b)
        sim_mono = cosine_sim(va_mono, vb_mono)

        # Multilingual
        va_multi = embed_multi.embed_query(text_a)
        vb_multi = embed_multi.embed_query(text_b)
        sim_multi = cosine_sim(va_multi, vb_multi)

        mono_scores.append(sim_mono)
        multi_scores.append(sim_multi)

        delta = sim_multi - sim_mono
        print(f"  {idx:<6} {lang_pair:<10} {sim_mono:<14.3f} {sim_multi:<14.3f} {delta:>+7.3f}")

print(f"\n  AVERAGES:")
print(f"    Monolingual:  {np.mean(mono_scores):.3f}")
print(f"    Multilingual: {np.mean(multi_scores):.3f}")
print(f"    Improvement:  {np.mean(multi_scores) - np.mean(mono_scores):+.3f}")
print(f"\n  Multilingual embeddings align same-meaning text across languages.")
print(f"  Monolingual embeddings treat foreign text as noise.")

## 10. Semantic vs Lexical — The Embedding Space

Verify that multilingual embeddings place translations close together but unrelated text far apart.

In [ ]:
# Positive pairs (same meaning, different languages) vs negative pairs (different meaning)
test_texts = [
    ("The server crashed at midnight.",            "en"),
    ("El servidor se cayó a medianoche.",           "es"),  # Same meaning
    ("Le serveur a planté à minuit.",               "fr"),  # Same meaning
    ("The new employee handbook was published.",    "en"),  # Different topic
    ("Las ventas aumentaron este mes.",             "es"),  # Different topic
]

vecs = [embed_multi.embed_query(t) for t, _ in test_texts]

print("PAIRWISE SIMILARITY (multilingual embeddings)")
print(f"{'':>45}", end="")
for i, (t, lang) in enumerate(test_texts):
    label = f"{lang}:{t[:15]}"
    print(f"{label:>18}", end="")
print()

for i, (t_i, lang_i) in enumerate(test_texts):
    label = f"{lang_i}: {t_i[:40]}"
    print(f"  {label:<43}", end="")
    for j, (t_j, lang_j) in enumerate(test_texts):
        s = cosine_sim(vecs[i], vecs[j])
        marker = "" if i == j else " *" if s > 0.7 else ""
        print(f"{s:>17.3f}{'' if not marker else marker}", end="")
    print()

print(f"\n  * = high similarity (>0.7)")
print(f"  Translations cluster together; unrelated text stays separate.")

---

# Part B — Multilingual Document Corpus

## 11. Build a Trilingual Knowledge Base

A company knowledge base with documents in English, Spanish, and French. Some documents are translations of the same content; others are language-exclusive.

In [ ]:
DOCUMENTS = [
    # --- English documents ---
    {"id": "EN01", "lang": "en", "topic": "remote_work",
     "title": "Remote Work Policy",
     "text": "Employees may work remotely up to 4 days per week. One mandatory in-office day per week for team collaboration. Remote workers receive a $1,000 annual home office stipend. Core hours are 10 AM to 3 PM in the employee's local timezone. VPN is required for all remote access to company systems."},

    {"id": "EN02", "lang": "en", "topic": "security",
     "title": "Information Security Policy",
     "text": "Multi-factor authentication is mandatory for all accounts. Passwords must be at least 16 characters. USB drives are prohibited — use approved cloud storage only. Security awareness training is conducted quarterly. All laptops must have full-disk encryption enabled. Data classification levels: Public, Internal, Confidential, Restricted."},

    {"id": "EN03", "lang": "en", "topic": "benefits",
     "title": "Employee Benefits Overview",
     "text": "Health insurance covers medical, dental, and vision for employees and dependents. The company matches 401(k) contributions up to 6% of salary. Unlimited PTO policy with a minimum of 15 days encouraged. Parental leave: 16 weeks paid for all parents. Professional development budget: $3,000 per year. Gym membership reimbursement up to $100 per month."},

    {"id": "EN04", "lang": "en", "topic": "onboarding",
     "title": "New Employee Onboarding Guide",
     "text": "Week 1: orientation, IT setup, and meet-the-team sessions. Week 2: product deep-dives and shadowing. Week 3-4: first assigned tasks with mentor support. 30-day check-in with manager. 90-day review with goals assessment. All new hires receive a buddy from another team for cross-functional context."},

    {"id": "EN05", "lang": "en", "topic": "api",
     "title": "API Rate Limiting Documentation",
     "text": "API rate limits: Free tier gets 100 requests per minute. Pro tier gets 1,000 requests per minute with burst to 2,000. Enterprise tier gets 10,000 requests per minute. Rate limit headers: X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Reset. When rate limited, the API returns HTTP 429 with a Retry-After header."},

    # --- Spanish documents ---
    {"id": "ES01", "lang": "es", "topic": "remote_work",
     "title": "Política de Trabajo Remoto",
     "text": "Los empleados pueden trabajar de forma remota hasta 4 días por semana. Un día obligatorio en la oficina por semana para colaboración del equipo. Los trabajadores remotos reciben un estipendio anual de $1,000 para oficina en casa. Las horas centrales son de 10 AM a 3 PM en la zona horaria local del empleado. Se requiere VPN para todo acceso remoto a los sistemas de la empresa."},

    {"id": "ES02", "lang": "es", "topic": "security",
     "title": "Política de Seguridad de la Información",
     "text": "La autenticación multifactor es obligatoria para todas las cuentas. Las contraseñas deben tener al menos 16 caracteres. Los dispositivos USB están prohibidos — use solo almacenamiento en la nube aprobado. La capacitación en seguridad se realiza trimestralmente. Todos los portátiles deben tener cifrado de disco completo. Niveles de clasificación de datos: Público, Interno, Confidencial, Restringido."},

    {"id": "ES03", "lang": "es", "topic": "compliance",
     "title": "Guía de Cumplimiento RGPD",
     "text": "El Reglamento General de Protección de Datos se aplica a todos los datos de ciudadanos europeos. Los datos personales deben almacenarse en servidores dentro de la Unión Europea. Las solicitudes de acceso a datos deben responderse en un plazo de 30 días. Se requiere consentimiento explícito para el procesamiento de datos. Las brechas de datos deben notificarse a las autoridades en 72 horas. Se debe designar un Delegado de Protección de Datos."},

    {"id": "ES04", "lang": "es", "topic": "performance",
     "title": "Proceso de Evaluación del Desempeño",
     "text": "Las evaluaciones de desempeño se realizan semestralmente en junio y diciembre. Los empleados completan una autoevaluación antes de la reunión. Los gerentes proporcionan retroalimentación escrita sobre objetivos, competencias y desarrollo. Las calificaciones van de 1 (no cumple) a 5 (excepcional). Los aumentos salariales se determinan por calificación: 3% para cumple, 5-8% para supera, 10-15% para excepcional. Los planes de desarrollo se crean en colaboración."},

    # --- French documents ---
    {"id": "FR01", "lang": "fr", "topic": "remote_work",
     "title": "Politique de Télétravail",
     "text": "Les employés peuvent travailler à distance jusqu'à 4 jours par semaine. Un jour obligatoire au bureau par semaine pour la collaboration d'équipe. Les télétravailleurs reçoivent une allocation annuelle de 1 000 $ pour le bureau à domicile. Les heures centrales sont de 10h à 15h dans le fuseau horaire local de l'employé. Le VPN est requis pour tout accès distant aux systèmes de l'entreprise."},

    {"id": "FR02", "lang": "fr", "topic": "security",
     "title": "Politique de Sécurité de l'Information",
     "text": "L'authentification multifacteur est obligatoire pour tous les comptes. Les mots de passe doivent comporter au moins 16 caractères. Les clés USB sont interdites — utilisez uniquement le stockage cloud approuvé. La formation à la sécurité est effectuée chaque trimestre. Tous les ordinateurs portables doivent avoir le chiffrement complet du disque activé. Niveaux de classification des données : Public, Interne, Confidentiel, Restreint."},

    {"id": "FR03", "lang": "fr", "topic": "benefits",
     "title": "Avantages Sociaux des Employés",
     "text": "L'assurance santé couvre les soins médicaux, dentaires et optiques pour les employés et leurs personnes à charge. L'entreprise verse une contribution équivalente au plan d'épargne retraite jusqu'à 6% du salaire. Politique de congés illimités avec un minimum de 15 jours encouragé. Congé parental : 16 semaines payées pour tous les parents. Budget de développement professionnel : 3 000 $ par an. Remboursement d'abonnement sportif jusqu'à 100 $ par mois."},

    {"id": "FR04", "lang": "fr", "topic": "incidents",
     "title": "Procédure de Gestion des Incidents",
     "text": "Les incidents sont classés en quatre niveaux de gravité : Critique (panne totale), Majeur (dégradation significative), Mineur (impact limité), et Informatif (aucun impact utilisateur). Pour les incidents critiques, l'équipe d'astreinte doit être contactée dans les 5 minutes. Communication aux clients dans les 30 minutes. Analyse post-incident obligatoire dans les 5 jours ouvrables. Culture sans blame — focus sur l'amélioration des systèmes."},
]

print(f"Corpus: {len(DOCUMENTS)} documents")
print(f"Languages: {dict(Counter(d['lang'] for d in DOCUMENTS))}")
print(f"Topics: {dict(Counter(d['topic'] for d in DOCUMENTS))}")
print(f"\nCross-lingual pairs (same topic in multiple languages):")
for topic in sorted(set(d['topic'] for d in DOCUMENTS)):
    docs = [d for d in DOCUMENTS if d['topic'] == topic]
    langs = [d['lang'] for d in docs]
    if len(langs) > 1:
        print(f"  {topic:<14} → {', '.join(langs)}")

## 12. Index with Multilingual Embeddings

In [ ]:
chroma_client = chromadb.Client()

# --- Multilingual index ---
try:
    chroma_client.delete_collection("multi")
except Exception:
    pass

col_multi = chroma_client.create_collection(name="multi", metadata={"hnsw:space": "cosine"})

texts = [d["text"] for d in DOCUMENTS]
metas = [
    {"doc_id": d["id"], "lang": d["lang"], "topic": d["topic"], "title": d["title"]}
    for d in DOCUMENTS
]
ids = [d["id"] for d in DOCUMENTS]

vecs_multi = embed_multi.embed_documents(texts)
col_multi.add(documents=texts, embeddings=vecs_multi, metadatas=metas, ids=ids)

# --- Monolingual index (baseline comparison) ---
try:
    chroma_client.delete_collection("mono")
except Exception:
    pass

col_mono = chroma_client.create_collection(name="mono", metadata={"hnsw:space": "cosine"})
vecs_mono = embed_mono.embed_documents(texts)
col_mono.add(documents=texts, embeddings=vecs_mono, metadatas=metas, ids=ids)

print(f"Multilingual index: {col_multi.count()} docs")
print(f"Monolingual index: {col_mono.count()} docs (baseline)")

---

# Part C — Cross-Lingual Retrieval

## 13. Retrieval Functions

In [ ]:
def retrieve(query: str, collection, embedder, top_k: int = TOP_K,
             lang_filter: str | None = None) -> list[dict]:
    """Retrieve from a collection using the given embedder."""
    qv = embedder.embed_query(query)
    where = {"lang": lang_filter} if lang_filter else None

    results = collection.query(
        query_embeddings=[qv], n_results=top_k, where=where,
    )

    hits = []
    for i in range(len(results["documents"][0])):
        meta = results["metadatas"][0][i]
        hits.append({
            "doc_id": meta["doc_id"],
            "lang": meta["lang"],
            "topic": meta["topic"],
            "title": meta["title"],
            "text": results["documents"][0][i],
            "score": round(1 - results["distances"][0][i], 4),
        })
    return hits


def print_hits(hits: list[dict], label: str = ""):
    if label:
        print(f"  {label}:")
    for i, h in enumerate(hits, 1):
        print(f"    {i}. [{h['doc_id']}] lang={h['lang']}  score={h['score']:.3f}  {h['title'][:40]}")


print("Retrieval functions ready")

## 14. Cross-Lingual Retrieval Demo

Query in one language, find documents in a different language.

In [ ]:
cross_lingual_queries = [
    # Spanish query → should find EN/FR remote work docs
    ("¿Cuántos días puedo trabajar desde casa?", "es", "remote_work"),
    # French query → should find EN/ES security docs
    ("Quelles sont les exigences de mot de passe?", "fr", "security"),
    # English query → should find ES/FR equivalents too
    ("What is the parental leave policy?", "en", "benefits"),
    # Spanish query → should find FR incident doc
    ("¿Cómo se manejan los incidentes críticos?", "es", "incidents"),
]

print("CROSS-LINGUAL RETRIEVAL")
print("=" * 90)

for query, query_lang, expected_topic in cross_lingual_queries:
    print(f"\n  Q [{query_lang}]: {query}")
    print(f"  Expected topic: {expected_topic}")

    # Multilingual retrieval
    hits_multi = retrieve(query, col_multi, embed_multi, top_k=3)
    print_hits(hits_multi, "Multilingual")

    # Monolingual retrieval (baseline)
    hits_mono = retrieve(query, col_mono, embed_mono, top_k=3)
    print_hits(hits_mono, "Monolingual (baseline)")

    # Check: did multilingual find the right topic across languages?
    multi_topics = [h["topic"] for h in hits_multi]
    mono_topics = [h["topic"] for h in hits_mono]
    multi_match = expected_topic in multi_topics
    mono_match = expected_topic in mono_topics
    print(f"  Topic found: multi={multi_match}, mono={mono_match}")

## 15. Language-Filtered Retrieval

Sometimes you want to search across languages but only retrieve docs in a specific language.

In [ ]:
print("LANGUAGE-FILTERED RETRIEVAL")
print("=" * 80)

query_es = "¿Qué beneficios tienen los empleados?"
print(f"\nQ [es]: {query_es}")

# Search all languages
hits_all = retrieve(query_es, col_multi, embed_multi, top_k=4)
print_hits(hits_all, "All languages")

# Filter to English only
hits_en = retrieve(query_es, col_multi, embed_multi, top_k=3, lang_filter="en")
print_hits(hits_en, "English only")

# Filter to French only
hits_fr = retrieve(query_es, col_multi, embed_multi, top_k=3, lang_filter="fr")
print_hits(hits_fr, "French only")

print(f"\n  Spanish query found relevant English and French documents")
print(f"  because multilingual embeddings share the same vector space.")

---

# Part D — Answer Language Control

## 16. Multilingual Answer Generation

The LLM reads documents in whatever language they were retrieved in, then answers in the **user's preferred language**.

In [ ]:
LANG_NAMES = {"en": "English", "es": "Spanish", "fr": "French"}

ANSWER_SYSTEM_TEMPLATE = """You answer questions using the provided evidence documents.
IMPORTANT: You MUST answer in {answer_lang}.
The documents may be in different languages — read and understand them all,
but always write your answer in {answer_lang}.
Cite documents by their ID: [EN01], [ES03], etc.
Include the document's language in the citation if it differs from the answer language."""


def detect_language(text: str) -> str:
    """Simple heuristic language detection based on common words."""
    text_lower = text.lower()
    es_markers = ["qué", "cuál", "cómo", "para", "los", "las", "del", "una", "por", "está"]
    fr_markers = ["quel", "comment", "les", "des", "une", "est", "pour", "dans", "aux", "sont"]
    es_score = sum(1 for w in es_markers if w in text_lower)
    fr_score = sum(1 for w in fr_markers if w in text_lower)
    if es_score > fr_score and es_score >= 2:
        return "es"
    if fr_score > es_score and fr_score >= 2:
        return "fr"
    return "en"


def cross_lingual_qa(
    question: str,
    answer_lang: str | None = None,
    verbose: bool = True,
) -> dict:
    """Full pipeline: detect language → retrieve (multilingual) → answer in target language."""
    t0 = time.time()

    # Detect query language (default answer language = query language)
    query_lang = detect_language(question)
    if not answer_lang:
        answer_lang = query_lang

    # Retrieve using multilingual embeddings (all languages)
    hits = retrieve(question, col_multi, embed_multi, top_k=3)

    # Format context with language annotations
    context_parts = []
    for h in hits:
        lang_label = LANG_NAMES.get(h['lang'], h['lang'])
        context_parts.append(
            f"[{h['doc_id']}] (Language: {lang_label}) {h['title']}\n{h['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # Generate answer
    answer_lang_name = LANG_NAMES.get(answer_lang, answer_lang)
    system = ANSWER_SYSTEM_TEMPLATE.format(answer_lang=answer_lang_name)
    prompt = f"QUESTION: {question}\n\nDOCUMENTS:\n{context}\n\nAnswer in {answer_lang_name}."

    answer = ask(prompt, system=system, temperature=0.2)
    elapsed = time.time() - t0

    result = {
        "question": question,
        "query_lang": query_lang,
        "answer_lang": answer_lang,
        "answer": answer,
        "hits": hits,
        "doc_langs": [h["lang"] for h in hits],
        "time_s": elapsed,
    }

    if verbose:
        print(f"  Query language: {query_lang}, Answer language: {answer_lang}")
        for h in hits:
            print(f"    [{h['doc_id']}] lang={h['lang']}  score={h['score']:.3f}  {h['title'][:40]}")

    return result


print("Cross-lingual QA pipeline ready")

## 17. Demo — Cross-Lingual QA

In [ ]:
demo_cases = [
    # Ask in Spanish → answer in Spanish (retrieves EN/ES/FR docs)
    ("¿Cuántos días puedo trabajar desde casa por semana?", None),
    # Ask in French → answer in French
    ("Quelles sont les règles de sécurité pour les mots de passe?", None),
    # Ask in English → force answer in Spanish
    ("What are the employee benefits for health insurance and retirement?", "es"),
    # Ask in Spanish → force answer in French
    ("¿Cómo se gestionan los incidentes de seguridad?", "fr"),
]

print("CROSS-LINGUAL QA DEMO")
print("=" * 100)

for question, forced_lang in demo_cases:
    q_lang = detect_language(question)
    a_lang = forced_lang or q_lang
    print(f"\nQ [{q_lang}]: {question}")
    if forced_lang:
        print(f"  Forced answer language: {LANG_NAMES[forced_lang]}")
    print("-" * 85)
    r = cross_lingual_qa(question, answer_lang=forced_lang)
    print(f"  Answer [{r['answer_lang']}]:")
    wrap_print("    " + r["answer"][:350])
    print(f"  Sources: {r['doc_langs']} | Time: {r['time_s']:.1f}s")

---

# Part E — Evaluation

## 18. Evaluation Setup

Test cross-lingual retrieval accuracy across all language pair combinations.

In [ ]:
EVAL_QUERIES = [
    # Remote work — exists in EN, ES, FR
    {"query_en": "How many days can I work from home?",
     "query_es": "¿Cuántos días puedo trabajar desde casa?",
     "query_fr": "Combien de jours puis-je travailler depuis chez moi?",
     "expected_topic": "remote_work"},

    # Security — exists in EN, ES, FR
    {"query_en": "What are the password requirements?",
     "query_es": "¿Cuáles son los requisitos de contraseña?",
     "query_fr": "Quelles sont les exigences de mot de passe?",
     "expected_topic": "security"},

    # Benefits — exists in EN, FR
    {"query_en": "What health insurance benefits are available?",
     "query_es": "¿Qué beneficios de seguro de salud están disponibles?",
     "query_fr": "Quels sont les avantages d'assurance santé disponibles?",
     "expected_topic": "benefits"},

    # GDPR — exists only in ES
    {"query_en": "What are the GDPR compliance requirements?",
     "query_es": "¿Cuáles son los requisitos de cumplimiento del RGPD?",
     "query_fr": "Quelles sont les exigences de conformité RGPD?",
     "expected_topic": "compliance"},

    # Incidents — exists only in FR
    {"query_en": "How are critical incidents handled?",
     "query_es": "¿Cómo se manejan los incidentes críticos?",
     "query_fr": "Comment les incidents critiques sont-ils gérés?",
     "expected_topic": "incidents"},

    # Performance — exists only in ES
    {"query_en": "How does the performance review process work?",
     "query_es": "¿Cómo funciona el proceso de evaluación del desempeño?",
     "query_fr": "Comment fonctionne le processus d'évaluation des performances?",
     "expected_topic": "performance"},
]

print(f"Evaluation: {len(EVAL_QUERIES)} query sets × 3 languages = {len(EVAL_QUERIES) * 3} retrieval tests")

## 19. Retrieval Accuracy — Monolingual vs Multilingual

In [ ]:
print("RETRIEVAL ACCURACY")
print("=" * 90)

results_table = []

for item in EVAL_QUERIES:
    topic = item["expected_topic"]

    for query_lang in ["en", "es", "fr"]:
        query = item[f"query_{query_lang}"]

        # Multilingual retrieval
        hits_multi = retrieve(query, col_multi, embed_multi, top_k=3)
        multi_topics = [h["topic"] for h in hits_multi]
        multi_hit = topic in multi_topics
        multi_rank = (multi_topics.index(topic) + 1) if multi_hit else None
        multi_top_score = hits_multi[0]["score"] if hits_multi else 0

        # Monolingual retrieval
        hits_mono = retrieve(query, col_mono, embed_mono, top_k=3)
        mono_topics = [h["topic"] for h in hits_mono]
        mono_hit = topic in mono_topics
        mono_rank = (mono_topics.index(topic) + 1) if mono_hit else None
        mono_top_score = hits_mono[0]["score"] if hits_mono else 0

        results_table.append({
            "topic": topic,
            "query_lang": query_lang,
            "multi_hit": multi_hit,
            "multi_rank": multi_rank,
            "multi_score": multi_top_score,
            "mono_hit": mono_hit,
            "mono_rank": mono_rank,
            "mono_score": mono_top_score,
        })

# Summary by query language
print(f"{'Query Lang':<12} {'Multi Hit%':<12} {'Mono Hit%':<12} {'Multi @1':<10} {'Mono @1':<10}")
print("-" * 56)

for lang in ["en", "es", "fr"]:
    items = [r for r in results_table if r["query_lang"] == lang]
    multi_hit_pct = sum(1 for r in items if r["multi_hit"]) / len(items)
    mono_hit_pct = sum(1 for r in items if r["mono_hit"]) / len(items)
    multi_at1 = sum(1 for r in items if r["multi_rank"] == 1) / len(items)
    mono_at1 = sum(1 for r in items if r["mono_rank"] == 1) / len(items)
    print(f"  {lang:<10} {multi_hit_pct:<12.0%} {mono_hit_pct:<12.0%} {multi_at1:<10.0%} {mono_at1:<10.0%}")

# Overall
multi_total = sum(1 for r in results_table if r["multi_hit"])
mono_total = sum(1 for r in results_table if r["mono_hit"])
n = len(results_table)
print(f"  {'TOTAL':<10} {multi_total/n:<12.0%} {mono_total/n:<12.0%}")
print(f"\n  Multilingual embeddings dramatically outperform monolingual")
print(f"  when queries and documents are in different languages.")

## 20. Cross-Lingual Heat Map

How well does retrieval work for each (query language → document language) pair?

In [ ]:
# For each query, check which document language was retrieved at rank 1
print("CROSS-LINGUAL RETRIEVAL QUALITY BY LANGUAGE PAIR")
print("(average top-1 similarity score)")
print("=" * 60)

pair_scores = {}
for q_lang in ["en", "es", "fr"]:
    for d_lang in ["en", "es", "fr"]:
        pair_scores[(q_lang, d_lang)] = []

for item in EVAL_QUERIES:
    for q_lang in ["en", "es", "fr"]:
        query = item[f"query_{q_lang}"]
        # Retrieve all, then check score by doc language
        hits = retrieve(query, col_multi, embed_multi, top_k=len(DOCUMENTS))
        for h in hits:
            if h["topic"] == item["expected_topic"]:
                pair_scores[(q_lang, h["lang"])].append(h["score"])

# Print as matrix
print(f"\n{'Query \\ Doc':<12}", end="")
for d_lang in ["en", "es", "fr"]:
    print(f"{d_lang:>10}", end="")
print()
print("-" * 42)

for q_lang in ["en", "es", "fr"]:
    print(f"  {q_lang:<10}", end="")
    for d_lang in ["en", "es", "fr"]:
        scores = pair_scores[(q_lang, d_lang)]
        avg = np.mean(scores) if scores else 0
        print(f"{avg:>10.3f}", end="")
    print()

print(f"\n  Same-language pairs (diagonal) should score highest.")
print(f"  Cross-lingual pairs should still be >0.5 for good retrieval.")

## 21. Answer Quality — Cross-Lingual vs Same-Language

In [ ]:
JUDGE_SYSTEM = """You evaluate if an answer correctly addresses the question using
information from the source documents.
The answer may be in a different language than the question.
Focus on factual accuracy, not language quality."""

JUDGE_PROMPT = """QUESTION: {question}

ANSWER (in {answer_lang}):
{answer}

EXPECTED FACT: The answer should mention: {expected_fact}

Evaluate: is the answer accurate and complete?
Return ONLY JSON:
{{"accurate": true/false,
  "contains_expected_fact": true/false,
  "quality": 1-5,
  "reasoning": "brief"}}"""

# Test key scenarios
test_scenarios = [
    {"question": "¿Cuántos días puedo trabajar desde casa?",
     "answer_lang": "es",
     "expected_fact": "4 days per week / 4 días por semana"},
    {"question": "Quelles sont les exigences de mot de passe?",
     "answer_lang": "fr",
     "expected_fact": "16 characters minimum / 16 caractères"},
    {"question": "What is the parental leave duration?",
     "answer_lang": "en",
     "expected_fact": "16 weeks paid"},
    {"question": "What are the GDPR data breach notification rules?",
     "answer_lang": "en",
     "expected_fact": "72 hours"},
]

print("CROSS-LINGUAL ANSWER QUALITY")
print("=" * 90)

for sc in test_scenarios:
    q_lang = detect_language(sc["question"])
    r = cross_lingual_qa(sc["question"], answer_lang=sc["answer_lang"], verbose=False)

    judge_result = parse_json(ask(
        JUDGE_PROMPT.format(
            question=sc["question"],
            answer_lang=LANG_NAMES[sc["answer_lang"]],
            answer=r["answer"][:350],
            expected_fact=sc["expected_fact"],
        ),
        system=JUDGE_SYSTEM, temperature=0.0,
    )) or {"accurate": False, "quality": 0}

    acc = "OK" if judge_result.get("accurate") else "--"
    qual = judge_result.get("quality", "?")
    fact = "OK" if judge_result.get("contains_expected_fact") else "--"
    doc_langs = ",".join(r["doc_langs"])

    print(f"  [{acc}] Q:{q_lang}→A:{sc['answer_lang']}  docs={doc_langs:<10} quality={qual}/5 fact={fact}")
    print(f"       Q: {sc['question'][:60]}")
    print(f"       A: {r['answer'][:80]}...")
    print()

---

# Part F — Advanced Topics

## 22. Same-Language Retrieval — Does Multilingual Hurt?

A common concern: does using a multilingual model reduce quality for same-language (e.g., English→English) retrieval?

In [ ]:
# Compare same-language retrieval quality
en_queries = [
    ("What are the password and MFA requirements?", "security"),
    ("How does the remote work policy work?", "remote_work"),
    ("What employee benefits are offered?", "benefits"),
    ("What is the onboarding process for new employees?", "onboarding"),
    ("What are the API rate limits?", "api"),
]

print("SAME-LANGUAGE (EN→EN) RETRIEVAL COMPARISON")
print("=" * 75)
print(f"{'Query':<50} {'Multi':<8} {'Mono':<8} {'Delta':>6}")
print("-" * 72)

multi_en_scores = []
mono_en_scores = []

for query, expected_topic in en_queries:
    hits_multi = retrieve(query, col_multi, embed_multi, top_k=3, lang_filter="en")
    hits_mono = retrieve(query, col_mono, embed_mono, top_k=3, lang_filter="en")

    # Find score for expected topic
    multi_score = next((h["score"] for h in hits_multi if h["topic"] == expected_topic), 0)
    mono_score = next((h["score"] for h in hits_mono if h["topic"] == expected_topic), 0)

    multi_en_scores.append(multi_score)
    mono_en_scores.append(mono_score)

    delta = multi_score - mono_score
    print(f"  {query[:48]:<50} {multi_score:<8.3f} {mono_score:<8.3f} {delta:>+5.3f}")

print(f"\n  Average: multi={np.mean(multi_en_scores):.3f}, mono={np.mean(mono_en_scores):.3f}")
print(f"  Delta: {np.mean(multi_en_scores) - np.mean(mono_en_scores):+.3f}")
print(f"\n  Small quality cost for same-language retrieval is expected.")
print(f"  The trade-off is worth it for cross-lingual capability.")

## 23. Full QA Demo — All Language Combinations

In [ ]:
# Take one question, answer in all 3 languages
base_question = {
    "en": "What is the remote work policy?",
    "es": "¿Cuál es la política de trabajo remoto?",
    "fr": "Quelle est la politique de télétravail?",
}

print("FULL LANGUAGE MATRIX — Same question, all combinations")
print("=" * 100)

for q_lang in ["en", "es", "fr"]:
    for a_lang in ["en", "es", "fr"]:
        question = base_question[q_lang]
        r = cross_lingual_qa(question, answer_lang=a_lang, verbose=False)

        doc_langs = ",".join(r["doc_langs"])
        print(f"\n  [{q_lang}→{a_lang}] docs={doc_langs}")
        print(f"    Q: {question[:55]}")
        print(f"    A: {r['answer'][:120]}...")

## 24. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Using monolingual embeddings for multilingual corpus | Cross-lingual similarity near zero; retrieval fails | Use `paraphrase-multilingual-MiniLM-L12-v2` or similar |
| Translating queries before embedding | Adds latency, translation errors, loses nuance | Multilingual embeddings handle this natively |
| Translating all docs to English first | Loses language-specific nuance, expensive | Index docs in original language with multilingual embeddings |
| Not specifying answer language in the prompt | LLM may answer in the document's language not the user's | Always set system prompt: "Answer in {target_language}" |
| Assuming all multilingual models are equal | Model quality varies hugely per language and task | Benchmark on your specific language pairs |
| Ignoring same-language quality trade-off | Multilingual models are slightly worse for same-language | Acceptable trade-off; hybrid approach possible for critical cases |

## 25. Production Improvement Ideas

1. **Hybrid embedding strategy** — use monolingual model for same-language queries, multilingual for cross-lingual
2. **Language detection** — use a proper library (`langdetect`, `fasttext`) instead of heuristics
3. **Parallel indexing** — index each document with both mono and multilingual embeddings, retrieve from both
4. **Translation-augmented retrieval** — translate top results into the answer language before LLM synthesis
5. **Language-specific quality metrics** — track retrieval quality per language pair and alert on degradation
6. **User language preference** — store user's preferred answer language in their profile

## 26. Exercises

### Exercise 1
Add a **fourth language** (e.g., German or Portuguese) to the corpus. Add translated versions of 2-3 existing documents and test cross-lingual retrieval quality for the new language pair.

### Exercise 2
Build a **hybrid retriever**: for same-language queries use the monolingual model, for cross-lingual queries use the multilingual model. Implement automatic routing based on detected query language vs document language distribution.

### Exercise 3
Implement a **cross-lingual consistency checker**: ask the same question in all 3 languages, get answers in English, and use an LLM to check whether the answers are semantically equivalent. Flag inconsistencies where different language queries produce different facts.

### Mini Challenge
Build a **document translation detector**: when the system retrieves a document in a language different from the user's query, automatically generate a summary of that document in the user's language and present it alongside the original. This helps users verify the source material.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Corpus: {len(DOCUMENTS)} documents in {dict(Counter(d['lang'] for d in DOCUMENTS))}")
print(f"Topics: {len(set(d['topic'] for d in DOCUMENTS))} unique, "
      f"{sum(1 for t in set(d['topic'] for d in DOCUMENTS) if sum(1 for d in DOCUMENTS if d['topic'] == t) > 1)} cross-lingual")
print(f"Embedding models: {MULTILINGUAL_EMBED}, {MONOLINGUAL_EMBED}")
print()
print(f"Cross-lingual similarity improvement:")
print(f"  Monolingual avg: {np.mean(mono_scores):.3f}")
print(f"  Multilingual avg: {np.mean(multi_scores):.3f} ({np.mean(multi_scores) - np.mean(mono_scores):+.3f})")
print()
print(f"Retrieval accuracy (topic found in top 3):")
print(f"  Multilingual: {multi_total}/{n} ({multi_total/n:.0%})")
print(f"  Monolingual: {mono_total}/{n} ({mono_total/n:.0%})")
print()
print(f"Components built:")
print(f"  1. Multilingual embedding comparison   — mono vs multi similarity scores")
print(f"  2. Cross-lingual retriever             — query in A, find docs in B")
print(f"  3. Language-filtered retriever          — metadata-based language filtering")
print(f"  4. Cross-lingual QA pipeline            — auto language detection + answer control")
print(f"  5. Retrieval accuracy evaluation        — per language and cross-lingual")
print(f"  6. LLM-as-judge quality assessment      — answer accuracy across languages")

## 27. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Multilingual embeddings map different languages to the same vector space** — translations get similar vectors |
| 2 | **Monolingual embeddings fail catastrophically for cross-lingual retrieval** — similarity drops to near zero |
| 3 | **The trade-off is small** — multilingual models are slightly worse for same-language retrieval, but the cross-lingual gain is massive |
| 4 | **Answer language must be explicitly controlled** — tell the LLM which language to use in the system prompt |
| 5 | **Language detection is needed** — auto-detect the query language to set the default answer language |
| 6 | **Not all documents need translation** — multilingual embeddings let you index docs in their original language |
| 7 | **Metadata filters complement embeddings** — use `lang` filter when you want docs only in a specific language |
| 8 | **Cross-lingual retrieval unlocks knowledge silos** — documents written for one region become accessible to all |
| 9 | **Model choice matters** — `paraphrase-multilingual-MiniLM-L12-v2` covers 50+ languages with good quality |
| 10 | **Real-world multilingual RAG is a production necessity** — most global organizations have content in multiple languages |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*